# UniLab Walkthrough: PPO + Go1 Joystick + MuJoCo

This notebook walks you through training a quadruped robot to walk using reinforcement learning (RL).

**No robotics or RL background needed** - every step explains the "why".

| Step | What | You will learn |
|------|------|----------------|
| 1 | Preview config | What parameters does RL training need? |
| 2 | Smoke test | 1 iteration to verify the environment works |
| 3 | Full training | Complete training loop (optional) |
| 4 | Inspect artifacts | Model files, logs, metrics |
| 5 | Playback | Watch the robot perform its learned skill |

### Key concepts (30-second version)

- **PPO** (Proximal Policy Optimization): a popular RL algorithm that improves a policy through trial and error
- **Policy**: a neural network - input: robot state (joint angles, velocities), output: actions (joint torques)
- **MuJoCo**: a physics engine that simulates the robot in a virtual world
- **Reward**: a signal telling the algorithm how well it's doing - moving forward = good, falling = bad

## Prerequisites

> If you've already completed the Quick Demo steps in README.md, skip to Step 1.

**System requirement**: Python >= 3.10, < 3.14

```bash
# 1. Install uv (Python package manager)
curl -LsSf https://astral.sh/uv/install.sh | sh

# 2. Clone the repo
git clone <repo-url> && cd UniLab

# 3. Install dependencies
uv sync

# 4. Activate the virtual environment
source .venv/bin/activate

# 5. Verify the CLI works
unilab --help
```

> **Colab not supported**: This project requires a local MuJoCo runtime for physics simulation. Please run this notebook on a local machine.

---
## Step 1: Environment check

Make sure we're in the right directory and key files exist.

In [ ]:
from pathlib import Path

ROOT = Path(".").resolve()
checks = {
    "scripts/": (ROOT / "scripts").is_dir(),
    "conf/": (ROOT / "conf").is_dir(),
    "conf/ppo/task/go1_joystick_flat/mujoco.yaml": (
        ROOT / "conf" / "ppo" / "task" / "go1_joystick_flat" / "mujoco.yaml"
    ).is_file(),
}
for name, ok in checks.items():
    status = "OK" if ok else "MISSING"
    print(f"  [{status}] {name}")
assert all(checks.values()), f"Run this notebook from the UniLab project root. Current dir: {ROOT}"
print(f"\nProject root: {ROOT}")

---
## Step 2: Preview the training config

Before training, let's see what parameters Hydra will use. Think of this as "previewing the menu":
- Algorithm hyperparameters (learning rate, batch size, etc.)
- Environment settings (number of parallel envs, simulation timestep, etc.)
- Reward function weights

`--cfg job` tells Hydra to print the final merged config without actually running training.

In [ ]:
!uv run python scripts/train_rsl_rl.py task=go1_joystick_flat/mujoco --cfg job

---
## Step 3: Smoke test (1 iteration)

Before committing to a full training run, let's do **1 iteration** to make sure everything works.
This takes just a few seconds.

If this cell errors out, fix the environment issue before continuing.

`algo.max_iterations=1` tells PPO to stop after 1 update step.

In [ ]:
!uv run python scripts/train_rsl_rl.py task=go1_joystick_flat/mujoco algo.max_iterations=1

If you see output like `iteration 1/1 ... done`, the environment is working correctly.

---
## Step 4: Full training (optional)

Full training takes longer (minutes to hours depending on hardware).

If you just want to experience the workflow, skip this and use the smoke test checkpoint from Step 3.

Two equivalent ways to launch training:

In [ ]:
# Option 1: Unified CLI (recommended)
# !uv run unilab train --algo ppo --task go1_joystick_flat --sim mujoco

# Option 2: Direct script call (equivalent)
# !uv run python scripts/train_rsl_rl.py task=go1_joystick_flat/mujoco

---
## Step 5: Inspect training artifacts

After training, all outputs are saved under `logs/rsl_rl_ppo/`.
Each run creates a timestamped directory containing:
- `model_*.pt` - model checkpoints saved at different iterations
- `run_config.json` - the full config used for this run
- `run_summary.json` - training result summary (if available)

In [ ]:
import json

log_base = ROOT / "logs" / "rsl_rl_ppo"
task_dir = log_base / "Go1JoystickFlatTerrain"

if not task_dir.is_dir():
    print(f"Go1JoystickFlatTerrain directory not found. Listing {log_base}:")
    task_dir = log_base

if task_dir.is_dir():
    runs = sorted([d for d in task_dir.iterdir() if d.is_dir()], key=lambda p: p.name)
    if runs:
        latest = runs[-1]
        print(f"Latest run: {latest.name}\n")
        print("Contents:")
        for f in sorted(latest.iterdir()):
            if f.is_file():
                print(f"  {f.name:40s}  {f.stat().st_size:>10,} bytes")
            else:
                print(f"  {f.name}/")
    else:
        print("No run directories found. Run Step 3 or Step 4 first.")
else:
    print(f"Log directory does not exist: {log_base}")

### Read training config and summary

Let's peek inside the JSON files to see what config was used and how training went.

In [ ]:
if task_dir.is_dir() and runs:
    latest = runs[-1]
    for name in ["run_config.json", "run_summary.json"]:
        path = latest / name
        if path.is_file():
            data = json.loads(path.read_text())
            print(f"=== {name} ===")
            print(json.dumps(data, indent=2, ensure_ascii=False)[:2000])
            print()
        else:
            print(f"{name} not found in {latest.name}")

---
## Step 6: Playback - watch the robot perform

Now let's load the trained checkpoint and run the policy in "play only" mode.
The robot will execute its learned walking behavior in the simulator.

`training.play_only=true` tells the script to skip training and just run inference.
`algo.load_run=-1` means "load the most recent run".

In [ ]:
# Option 1: Unified CLI
!uv run unilab eval --algo ppo --task go1_joystick_flat --sim mujoco --load-run -1

# Option 2: Direct script call (equivalent)
# !uv run python scripts/train_rsl_rl.py task=go1_joystick_flat/mujoco training.play_only=true algo.load_run=-1

### Display the playback video

If playback generated a video file, we can display it right here in the notebook.

In [ ]:
from IPython.display import Video

if task_dir.is_dir() and runs:
    latest = runs[-1]
    videos = list(latest.glob("play_video*.mp4"))
    if videos:
        vid = videos[0]
        print(f"Video: {vid}")
        display(Video(str(vid), embed=True, width=640))
    else:
        print(f"No video found in {latest.name}.")
        print("Playback may not have generated a video, or rendering was unavailable.")
else:
    print("No run directory available. Complete training first.")

---
## What's next?

- **Try a different robot**: `uv run unilab train --algo ppo --task g1_walk_flat --sim mujoco`
- **Try a different algorithm**: `uv run unilab train --algo sac --task g1_walk_flat --sim mujoco`
- **Run the demo notebook**: see `demo.ipynb` for one-click checkpoint playback
- **Explore all options**: `uv run unilab train --help`

### How the unified CLI maps to scripts

| CLI command | Underlying script | Config path |
|-------------|-------------------|-------------|
| `unilab train --algo ppo` | `scripts/train_rsl_rl.py` | `conf/ppo/task/{task}/{sim}.yaml` |
| `unilab train --algo sac` | `scripts/train_offpolicy.py` | `conf/offpolicy/task/sac/{task}/{sim}.yaml` |
| `unilab train --algo appo` | `scripts/train_appo.py` | `conf/appo/task/{task}/{sim}.yaml` |
| `unilab eval ...` | Same as train + `training.play_only=true` | Same |
| `unilab demo` | Preset-based checkpoint playback | N/A |